In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')


from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,ConfusionMatrixDisplay





from sklearn.preprocessing import StandardScaler


from sklearn.ensemble import (RandomForestClassifier,

                              GradientBoostingClassifier,
)

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"


column_names = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]


df = pd.read_csv(url, names=column_names)





In [19]:
print("Shape:",df.shape)
df.head(10)

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [21]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [22]:
df.nunique()

,0
Pregnancies,17
Glucose,136
BloodPressure,47
SkinThickness,51
Insulin,186
BMI,248
DiabetesPedigreeFunction,517
Age,52
Outcome,2


In [23]:
print("\n missing values  :  \n",df.isnull().sum()[df.isnull().sum()>0])


 missing values  :  
 Series([], dtype: int64)


In [24]:
print("\n number of duplicate rows: ", df.duplicated().sum())


 number of duplicate rows:  0


In [25]:
#The value is not zero medically

zero_counts = (df == 0).sum()
zero_percentages = ((df == 0).sum() / len(df) * 100).round(2)
print('\n zer0 values summary(count,percentage) \n')
zero_summary = pd.DataFrame(
    {
    'Zero_Count': zero_counts,
    'zero_percentage %': zero_percentages
}
)
print(zero_summary)



 zer0 values summary(count,percentage) 

                          Zero_Count  zero_percentage %
Pregnancies                      111              14.45
Glucose                            5               0.65
BloodPressure                     35               4.56
SkinThickness                    227              29.56
Insulin                          374              48.70
BMI                               11               1.43
DiabetesPedigreeFunction           0               0.00
Age                                0               0.00
Outcome                          500              65.10


In [26]:
#Replacing the zero values
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_clean = df.copy()

for col in zero_cols:
    df_clean[col] = df_clean.groupby('Outcome')[col].transform(
        lambda group: group.replace(0, group[group != 0].median())
    )

print('\n number of remaining zero\n')
print((df_clean[zero_cols] == 0).sum())


 number of remaining zero

Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


In [27]:
#skew_comparison
skew_comparison = pd.DataFrame({
    'Original Skewness': df.skew(),
    'Cleaned Skewness': df_clean.skew()
})
print('\n Skewness Comparison \n')
print(skew_comparison.round(2))


 Skewness Comparison 

                          Original Skewness  Cleaned Skewness
Pregnancies                            0.90              0.90
Glucose                                0.17              0.53
BloodPressure                         -1.84              0.14
SkinThickness                          0.11              0.82
Insulin                                2.27              3.03
BMI                                   -0.43              0.61
DiabetesPedigreeFunction               1.92              1.92
Age                                    1.13              1.13
Outcome                                0.64              0.64


In [28]:
#Outlier detection_IQR method

df_final = df_clean.copy()
capped_summary = {}

for col in df_final.columns:
    if col == 'Outcome':
        continue
    Q1 = df_final[col].quantile(0.25)
    Q3 = df_final[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_capped = ((df_final[col] < lower_bound) | (df_final[col] > upper_bound)).sum()
    df_final[col] = df_final[col].clip(lower=lower_bound, upper=upper_bound)
    capped_summary[col] = n_capped

print('\n Outliers Capped Per Column (IQR bounds) \n')
print(pd.Series(capped_summary))

remaining = {}
for col in df_final.columns:
    if col == 'Outcome':
        continue
    Q1 = df_final[col].quantile(0.25)
    Q3 = df_final[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    remaining[col] = ((df_final[col] < lower_bound) | (df_final[col] > upper_bound)).sum()

print('\n number of remaining outliers after capping\n')
print(pd.Series(remaining))



 Outliers Capped Per Column (IQR bounds) 

Pregnancies                  4
Glucose                      0
BloodPressure               14
SkinThickness               87
Insulin                     51
BMI                          8
DiabetesPedigreeFunction    29
Age                          9
dtype: int64

 number of remaining outliers after capping

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64


In [29]:
#class distribution outcome
outcome_labels = df_clean['Outcome'].map({0: 'Non-Diabetic', 1: 'Diabetic' })

print('\n Outcome Class Distribution \n')
print(outcome_labels.value_counts())

print('\n Outcome Class Percentage (%) \n')
print((outcome_labels.value_counts(normalize=True) * 100).round(2))


 Outcome Class Distribution 

Outcome
Non-Diabetic    500
Diabetic        268
Name: count, dtype: int64

 Outcome Class Percentage (%) 

Outcome
Non-Diabetic    65.1
Diabetic        34.9
Name: proportion, dtype: float64


In [30]:
#correlation_with_outcome
correlation_with_outcome = df_clean.corr()['Outcome'].sort_values(ascending=False)
print("Correlation of each feature with Outcome:\n")
print(correlation_with_outcome.round(3))

Correlation of each feature with Outcome:

Outcome                     1.000
Glucose                     0.496
Insulin                     0.377
BMI                         0.316
SkinThickness               0.295
Age                         0.238
Pregnancies                 0.222
BloodPressure               0.174
DiabetesPedigreeFunction    0.174
Name: Outcome, dtype: float64


In [31]:
df_clean

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76.0,48,180.0,32.9,0.171,63,0
764,2,122,70.0,27,102.5,36.8,0.340,27,0
765,5,121,72.0,23,112.0,26.2,0.245,30,0
766,1,126,60.0,32,169.5,30.1,0.349,47,1


In [32]:

x=df_clean.drop('Outcome',axis=1)
y=df_clean['Outcome']

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

print(x_train.shape)
print(x_test.shape)

(614, 8)
(154, 8)


In [33]:
results= {}

def evaluate_model(model,x_tr,x_te,name):
    model.fit(x_tr,y_train)
    y_pred = model.predict(x_te)
    accuracy = accuracy_score(y_test,y_pred)
    results[name]=accuracy
    print(f"{name}: Accuracy = {accuracy:.3f}")
    return model

In [34]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)

log_reg.fit(x_train, y_train)

train_acc = log_reg.score(x_train, y_train)
test_acc = log_reg.score(x_test, y_test)
print(f"Train = {train_acc:.3f} | Test = {test_acc:.3f}")

Train = 0.796 | Test = 0.708


In [35]:
rf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=42)
rf.fit(x_train, y_train)
train_acc = rf.score(x_train, y_train)
test_acc = rf.score(x_test, y_test)
print(f"Train = {train_acc:.3f} | Test = {test_acc:.3f}")

Train = 0.932 | Test = 0.864


In [36]:
svm=SVC(kernel='linear',C=1,random_state=42)
svm.fit(x_train,y_train)
train_acc=svm.score(x_train,y_train)
test_acc=svm.score(x_test,y_test)
print(f"Train = {train_acc:.3f} | Test = {test_acc:.3f}")

Train = 0.811 | Test = 0.740


In [37]:
Gb=GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,max_depth=5, random_state=42)
Gb= evaluate_model(Gb,x_train,x_test,'GradientBoosting')

GradientBoosting: Accuracy = 0.896


In [38]:
y_pred= rf.predict(x_test)
print(classification_report(y_test,y_pred))


              precision    recall  f1-score   support

           0       0.89      0.90      0.90       100
           1       0.81      0.80      0.80        54

    accuracy                           0.86       154
   macro avg       0.85      0.85      0.85       154
weighted avg       0.86      0.86      0.86       154



In [39]:
print(confusion_matrix(y_test,y_pred))

[[90 10]
 [11 43]]
